# LLM-RecFusion — Stage 5: Listwise 生成式重排 Prompt 开发

**Objective**: 彻底抛弃传统 Pointwise 逐个打分的大模型重排范式，构建基于 LLM 的 Listwise 生成式排序 Prompt，让大模型一目十行，直接输出最优排序序号。

---

### 范式对比

```
传统 Pointwise 重排（已淘汰）:
  for item in candidates:          ← O(N) 次前向传播
      score = llm.score(item)     ← 每次独立打分，无交叉特征
  reranked = sort(candidates, by=score)

我们 Listwise 生成式重排（升维）:
  prompt = build_listwise_prompt(user_profile, candidates)  ← 1 次前向传播
  indices = llm.generate(prompt)                            ← O(1) 延迟
  reranked = [candidates[i] for i in indices]               ← Self-Attention 全局交叉
```

---

### Notebook 流程

```
┌──────────────────────────────────────────────────────────────────┐
│ 1. 模拟用户画像 & Top-5 候选物品（含不相关爆款，测试 LLM 重排能力） │
├──────────────────────────────────────────────────────────────────┤
│ 2. 核心 Prompt 构造器 build_listwise_prompt()                     │
│    角色设定 + 上下文注入 + 输出格式强制约束（防幻觉）             │
├──────────────────────────────────────────────────────────────────┤
│ 3. Mock 干跑验证 + 鲁棒正则解析器 parse_llm_output()              │
│    单测覆盖：正常输出 + 异常话痨输出 + 空输出                      │
├──────────────────────────────────────────────────────────────────┤
│ 4. 大厂风格工程结论（README 素材）                                 │
└──────────────────────────────────────────────────────────────────┘
```

---

## 1. 模拟精排输出与用户数据

构造模拟的用户历史偏好文本，以及精排模型输出的 Top-5 候选列表。

**对抗性测试设计**：在 5 个候选中混入与用户历史偏好不相关的爆款物品（如婴儿奶粉、美白面霜），检验 LLM 是否具备辨别能力——即不被全局爆款冲昏头脑，而是基于用户个性化偏好做排序。

In [ ]:
# ============================================================
# 1. 模拟用户画像与 Top-5 候选物品
# ============================================================

import json
import re
from typing import List, Dict, Optional

# ---------- 模拟用户画像 ----------
user_profile: str = (
    "该用户近期频繁浏览和购买：高配游戏本、机械键盘、人体工学椅、"
    "高刷新率电竞显示器、游戏鼠标。用户偏好明确：高性能电竞装备，"
    "客单价 2000~15000 元，对品牌和性能敏感，对非电竞类日用品不感兴趣。"
)

# ---------- 构造对抗性候选列表 ----------
# 故意混入 2 个不相关爆款，测试 LLM 重排能力
candidate_list: List[Dict[str, str]] = [
    {
        "id": 1,
        "name": "ROG 魔霸 7 Plus 高配游戏本",
        "category": "电脑/游戏本",
        "price": 12999,
        "desc": "RTX4070 + R9 7845HX + 17.3英寸 2.5K 240Hz 电竞屏"
    },
    {
        "id": 2,
        "name": "赫莲娜 黑白绷带面霜套装",
        "category": "美妆/护肤",
        "price": 3880,
        "desc": "玻色因修护抗老，贵妇级面霜，月销 10 万+ 爆款"
    },
    {
        "id": 3,
        "name": "罗技 G Pro X Superlight 无线游戏鼠标",
        "category": "外设/鼠标",
        "price": 1299,
        "desc": "63g 超轻量，HERO 25K 传感器，职业电竞选手同款"
    },
    {
        "id": 4,
        "name": "a2 至初 白金版幼儿配方奶粉 3段",
        "category": "母婴/奶粉",
        "price": 488,
        "desc": "新西兰原装进口，A2 蛋白呵护肠胃，月销 20 万+ 爆款"
    },
    {
        "id": 5,
        "name": "Herman Miller Aeron 人体工学椅",
        "category": "办公家具/人体工学椅",
        "price": 13999,
        "desc": "王者级人体工学椅，PostureFit SL 腰部支撑，8 年质保"
    },
]

print("=" * 60)
print("用户画像:")
print(user_profile)
print("=" * 60)
print("\n候选物品列表（序号 [1]~[5]）:")
for item in candidate_list:
    print(f"  [{item['id']}] {item['name']}  |  ¥{item['price']}  |  {item['desc']}")
print("=" * 60)
print("\n⚠ 对抗性分析：物品 #2 (面霜) 和 #4 (奶粉) 是跨域爆款，与用户偏好完全不相关。")
print("  理想重排结果应将这些物品排在末尾。")
print("=" * 60)

## 2. 核心 Prompt 构造器

设计三层次 Prompt 模板：
1. **角色设定** — 站在全局视角的大厂资深电商推荐专家
2. **上下文注入** — 用户画像 + 带序号的候选商品列表
3. **输出格式强制约束** — 防幻觉策略，仅输出序号数组

使用 Python 三引号多行 `f-string` 优雅拼接。

In [ ]:
# ============================================================
# 2. 核心 Prompt 构造器
# ============================================================


def build_listwise_prompt(
    user_profile: str,
    candidate_list: List[Dict[str, str]],
) -> str:
    """
    构造 Listwise 生成式重排 Prompt。

    Parameters
    ----------
    user_profile : str
        用户画像文本，描述用户的历史偏好和行为。
    candidate_list : List[Dict[str, str]]
        候选物品列表，每个物品至少包含 id, name, category, price, desc 字段。

    Returns
    -------
    str
        完整的 Prompt 字符串，可直接输入 LLM。
    """

    # ---------- 构建候选列表文本 ----------
    candidate_lines: List[str] = []
    for item in candidate_list:
        candidate_lines.append(
            f"[{item['id']}] {item['name']}\n"
            f"    品类: {item['category']} | 价格: ¥{item['price']}\n"
            f"    描述: {item['desc']}"
        )
    candidates_text: str = "\n".join(candidate_lines)

    # ---------- 三层次 Prompt 模板 ----------
    prompt: str = f"""\
## 角色设定
你是一个大厂资深电商推荐专家，请站在全局视角对以下推荐列表进行重排，以最大化用户的点击兴趣和列表多样性。

## 上下文注入
### 用户画像
{user_profile}

### 候选商品列表（输入）
{candidates_text}

## 输出格式强制约束
请严格按以下要求输出：
1. 禁止输出任何多余的解释、引言、评论或过渡语（例如“好的”、“我已为您排好序”等）。
2. **仅输出** 一个 Python 风格的数字列表，表示按推荐程度从高到低排列的序号。
3. 格式必须严格遵循：[3, 1, 4, 2, 5]
4. 序号范围必须在 [1, 2, 3, 4, 5] 内，且不允许重复和遗漏。

输出："""

    return prompt


# ---------- 实例化 ----------
prompt: str = build_listwise_prompt(user_profile, candidate_list)

print("=" * 70)
print("构建完成的 Prompt:")
print("=" * 70)
print(prompt)
print("=" * 70)

## 3. 轻量级干跑验证（Dry Run）与正则解析

由于当前阶段尚未部署 vLLM（任务三再做），我们先编写一个 Mock 函数模拟理想输出，再编写一个极简且鲁棒的正则提取解析器。

### 单测覆盖场景
- **正常输出**：LLM 精确输出 `[4, 1, 2, 5, 3]`
- **话痨输出**：LLM 在数组前后加了废话
- **空输出**：LLM 输出为空或完全不包含数组结构

In [ ]:
# ============================================================
# 3a. Mock 函数 — 模拟理想 LLM 输出
# ============================================================

import random


def mock_llm_generate(prompt: str) -> str:
    """
    模拟大模型接收 Prompt 后的理想输出。
    在实际部署中，此函数将被 vLLM 的 generate() 调用替换。

    Mock 逻辑：将强相关的电竞/游戏本/外设排在前面，不相关的排在后面。
    这模拟了一个理想 LLM 应有的排序能力。
    """
    # 理想重排：游戏区物品优先，非相关爆款置后
    # [1] ROG 游戏本  → 最相关
    # [3] 游戏鼠标   → 强相关
    # [5] 人体工学椅 → 中等相关（用户历史中有）
    # [2] 面霜       → 不相关
    # [4] 奶粉       → 不相关
    ideal_reranked: List[int] = [1, 3, 5, 2, 4]
    return str(ideal_reranked)


# ---------- 干跑验证 ----------
mock_output: str = mock_llm_generate(prompt)
print("Mock LLM 输出:")
print(mock_output)

In [ ]:
# ============================================================
# 3b. 鲁棒正则提取解析器
# ============================================================


def parse_llm_output(text: str) -> Optional[List[int]]:
    """
    从 LLM 输出中鲁棒地提取排序序号数组。

    无论大模型是否输出了一堆废话，只要文本中包含类似 [x, x, x, x, x]
    的结构，就能将其安全地解析为 Python List。

    正则策略：
      - 匹配方括号内逗号分隔的数字序列
      - 过滤掉非数字干扰字符
      - 支持任意长度的数组（泛化到 Top-N）

    防幻觉防御：
      - 如果完全没有匹配，返回 None 而非抛异常
      - 如果匹配了但数字不在有效范围内，由调用方校验

    Parameters
    ----------
    text : str
        LLM 输出的原始文本。

    Returns
    -------
    Optional[List[int]]
        解析出的序号列表，若无法解析则返回 None。
    """
    if not text or not text.strip():
        return None

    # 核心正则：匹配方括号内逗号分隔的数字
    # 解释：\[       → 左方括号
    #       \s*      → 空白字符（含换行）
    #       (\d+     → 捕获组：一个或多个数字
    #       (?:\s*,\s*\d+)*  → 逗号分隔的后续数字
    #       \s*      → 尾部空白
    #       \]       → 右方括号
    pattern = r"\[\s*(\d+(?:\s*,\s*\d+)*)\s*\]"
    matches: List[str] = re.findall(pattern, text)

    if not matches:
        return None

    # 提取数字部分，按逗号分割，转换为 int
    digits_str: str = matches[-1]
    parsed: List[int] = [int(x.strip()) for x in digits_str.split(",")]

    return parsed


def validate_rerank_indices(
    indices: Optional[List[int]],
    expected_len: int = 5,
) -> bool:
    """
    校验重排结果是否合法。

    检查项：
      - 不能为 None
      - 长度与候选列表一致
      - 包含所有 1..N 序号（无重复、无遗漏）
    """
    if indices is None:
        return False
    if len(indices) != expected_len:
        return False
    if sorted(indices) != list(range(1, expected_len + 1)):
        return False
    return True


# ---------- 解析 Mock 输出 ----------
parsed = parse_llm_output(mock_output)
is_valid = validate_rerank_indices(parsed, expected_len=5)
print(f"原始文本: {mock_output}")
print(f"解析结果: {parsed}")
print(f"校验通过: {is_valid}")

if is_valid:
    print("\n最终重排结果:")
    for idx, item_id in enumerate(parsed, 1):
        item = candidate_list[item_id - 1]
        print(f"  Top-{idx}: [{item['id']}] {item['name']}  ¥{item['price']}")

In [ ]:
# ============================================================
# 3c. 单测 — 覆盖多种真实场景
# ============================================================

print("=" * 60)
print("单测 1: 正常输出 — LLM 精确返回数组")
print("=" * 60)
test_normal = "[4, 1, 2, 5, 3]"
result = parse_llm_output(test_normal)
print(f"  输入: {test_normal!r}")
print(f"  解析: {result}")
print(f"  通过: {validate_rerank_indices(result)}")

print()
print("=" * 60)
print("单测 2: 话痨输出 — 大模型在前面/后面加了废话")
print("=" * 60)
test_verbose = (
    "好的，根据用户画像分析，该用户偏好电竞装备，因此我将面霜和奶粉排在末尾。"
    "具体的排序为：[1, 3, 5, 2, 4] 希望对您有帮助！"
)
result = parse_llm_output(test_verbose)
print(f"  输入: {test_verbose!r}")
print(f"  解析: {result}")
print(f"  通过: {validate_rerank_indices(result)}")

print()
print("=" * 60)
print("单测 3: 极废话输出 — 大模型话痨晚期，数组夹杂在长文中")
print("=" * 60)
test_very_verbose = (
    "好的，我已仔细分析了用户的购买历史和浏览记录，"
    "考虑到用户的偏好主要集中在高性能电竞领域，我给出的推荐排序是 "
    "[1, 3, 5, 2, 4] ，如有需要可以进一步调整。同时，"
    "我注意到奶粉和面霜的跨域购买可能性较低，因此排在最后。"
)
result = parse_llm_output(test_very_verbose)
print(f"  输入: {test_very_verbose!r}")
print(f"  解析: {result}")
print(f"  通过: {validate_rerank_indices(result)}")

print()
print("=" * 60)
print("单测 4: 异常话痨 — 模型在数组内/外都加了换行和空格")
print("=" * 60)
test_noisy = "输出：\n\n[  1,3,5,  2,4  ]\n\n以上是我的排序。"
result = parse_llm_output(test_noisy)
print(f"  输入: {test_noisy!r}")
print(f"  解析: {result}")
print(f"  通过: {validate_rerank_indices(result)}")

print()
print("=" * 60)
print("单测 5: 空输出 — 模型完全无法解析")
print("=" * 60)
test_empty = ""
result = parse_llm_output(test_empty)
print(f"  输入: {test_empty!r}")
print(f"  解析: {result}")
print(f"  通过: {result is None}")

print()
print("=" * 60)
print("单测 6: 乱码输出 — 模型胡言乱语不含数组结构")
print("=" * 60)
test_gibberish = "我不确定该如何排序，不过我觉得面霜挺好的。"
result = parse_llm_output(test_gibberish)
print(f"  输入: {test_gibberish!r}")
print(f"  解析: {result}")
print(f"  通过: {result is None}")

print()
print("=" * 60)
print("单测 7: 多数组干扰 — 模型在中间生成了中间过程数组")
print("=" * 60)
test_multi = (
    "我想先分析一下：第一步 [1,2,3,4,5] 是原始顺序，"
    "经过推理后最终排序为 [1,3,5,2,4]。"
)
result = parse_llm_output(test_multi)
print(f"  输入: {test_multi!r}")
print(f"  解析: {result}")
print(f"  通过: {validate_rerank_indices(result)}")

## 4. 极客硬核工程结论（README 素材）

---

### Pointwise → Listwise：范式升维的必然性

传统的 Pointwise 重排需要对 N 个候选物品进行 N 次大模型前向传播，引发 $O(N)$ 的延迟爆炸，线上完全不可用。而 Listwise 生成式范式只需 **1次** 前向传播，便能利用 LLM 的 Self-Attention 机制捕获商品间的交叉关系，实现延迟压榨与全局最优解的双赢。

### 本阶段关键产出

| 模块 | 文件 | 说明 |
|------|------|------|
| Prompt 构造器 | [`build_listwise_prompt()`](#2-核心-prompt-构造器) | 三层次模板：角色设定 + 上下文注入 + 输出约束 |
| 正则解析器 | [`parse_llm_output()`](#3b-鲁棒正则提取解析器) | 抗话痨、抗幻觉、抗乱码的鲁棒提取 |
| 结果校验器 | [`validate_rerank_indices()`](#3b-鲁棒正则提取解析器) | 长度 + 唯一性 + 范围三重校验 |

### 防幻觉体系设计

```
┌─ Prompt 层 ─────────────────────────────────────────────┐
│  "禁止输出任何多余解释...仅输出 [3, 1, 4, 2, 5]"        │
└──────────────────────────────────────────────────────────┘
                            ↓
┌─ 正则解析层 ─────────────────────────────────────────────┐
│  re.search(r"\[\s*(\d+(?:\s*,\s*\d+)*)\s*\]", text)   │
│  无论话痨多少废话，只要文本含 [x,x,...] 即可稳定提取     │
└──────────────────────────────────────────────────────────┘
                            ↓
┌─ 业务校验层 ─────────────────────────────────────────────┐
│  validate_rerank_indices()                              │
│  ├── 非 None 检查                                      │
│  ├── 长度匹配检查                                      │
│  └── 序号全排列检查 (1..N 无重复无遗漏)                   │
└──────────────────────────────────────────────────────────┘
```

### 下一步（任务二 ~ 任务三）

1. **任务二**：将本 Notebook 中的 Prompt 构造器 + 解析器封装为生产级 Python 模块 `rerank/listwise_reranker.py`
2. **任务三**：对接 vLLM 部署，用真实模型（如 Qwen2.5-7B-Instruct）替换 Mock 函数，接入线上推理管道
3. **性能目标**：5 个候选物品的端到端重排延迟 < 500ms（含网络 + 推理 + 解析）

---

*LLM-RecFusion — 从 Pointwise 到 Listwise，一次前向传播，全局最优重排。*